# Загрузка данных superstore_sales в PostgreSQL с использованием PySpark

Этот notebook читает файл superstore_sales.csv и загружает данные в таблицу public.superstore_sales в PostgreSQL через Docker Compose.

## 1. Подключение библиотек и настройка SparkSession

Импортируем необходимые библиотеки и создаем SparkSession с поддержкой JDBC для подключения к PostgreSQL.

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .remote("sc://spark-connect:15002") \
    .appName("LoadSuperstore") \
    .getOrCreate()

In [3]:
spark.conf.get("spark.driver.host")

'50e35f1e0481'

In [4]:
spark.conf.get("spark.ui.port")

'4041'

## 2. Чтение CSV файла с помощью PySpark

Используем `spark.read.csv` для чтения файла superstore_sales.csv с опциями header и inferSchema.

In [5]:
# Путь к CSV файлу (относительный путь от notebook)
csv_file_path = "/opt/spark/work-dir/data/superstore_sales.csv"

# Читаем CSV файл
df = spark.read.csv(
    csv_file_path,
    header=True,
    inferSchema=True,
    escape='"',
    multiLine=True
)

print(f"CSV file successfully loaded!")
print(f"Number of rows: {df.count()}")
print(f"Number of columns: {len(df.columns)}")

CSV file successfully loaded!
Number of rows: 9800
Number of columns: 18


## 3. Преобразование схемы и проверка данных

Проверяем схему DataFrame, показываем образец данных и выполняем необходимые преобразования типов.

In [6]:
# Проверяем схему DataFrame
print("DataFrame Schema:")
df.printSchema()

print("\n" + "="*80)
print("Sample Data (first 5 rows):")
df.show(5, truncate=False)

print("\n" + "="*80)
print("Column names:")
print(df.columns)

DataFrame Schema:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)


Sample Data (first 5 rows):
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-------------

## 4. Подключение к PostgreSQL через JDBC

Настраиваем параметры подключения к PostgreSQL контейнеру, запущенному через Docker Compose.

In [7]:
# Параметры подключения к PostgreSQL
postgres_host = "postgres"
postgres_port = 5432
postgres_db = "postgres"
postgres_user = "postgres"
postgres_password = "postgres"
postgres_table = "public.superstore_sales"

# Формируем JDBC URL
jdbc_url = f"jdbc:postgresql://{postgres_host}:{postgres_port}/{postgres_db}"

# Драйвер JDBC
jdbc_driver = "org.postgresql.Driver"

print(f"PostgreSQL Connection Settings:")
print(f"Host: {postgres_host}")
print(f"Port: {postgres_port}")
print(f"Database: {postgres_db}")
print(f"Table: {postgres_table}")
print(f"JDBC URL: {jdbc_url}")

PostgreSQL Connection Settings:
Host: postgres
Port: 5432
Database: postgres
Table: public.superstore_sales
JDBC URL: jdbc:postgresql://postgres:5432/postgres


## 5. Создание таблицы и загрузка данных в public.superstore_sales

Используем DataFrame.write.jdbc для записи данных в PostgreSQL. Параметр mode='overwrite' пересоздаст таблицу.

In [8]:
try:
    print(f"Starting data load to PostgreSQL...")
    print(f"Table: {postgres_table}")
    print(f"Total records to load: {df.count()}")
    
    # Загружаем данные в PostgreSQL
    df.write \
        .format("jdbc") \
        .mode("overwrite") \
        .option("url", jdbc_url) \
        .option("dbtable", postgres_table) \
        .option("user", postgres_user) \
        .option("password", postgres_password) \
        .option("driver", jdbc_driver) \
        .save()
    
    print(f"\n{'='*80}")
    print(f"✓ Data successfully loaded to PostgreSQL!")
    print(f"✓ Table: {postgres_table}")
    print(f"✓ Records loaded: {df.count()}")
    print(f"{'='*80}")
    
except Exception as e:
    print(f"Error during data load: {str(e)}")
    raise

Starting data load to PostgreSQL...
Table: public.superstore_sales
Total records to load: 9800

✓ Data successfully loaded to PostgreSQL!
✓ Table: public.superstore_sales
✓ Records loaded: 9800


## 6. Проверка загруженных данных

Читаем данные обратно из PostgreSQL для проверки загрузки.

In [9]:
try:
    # Читаем данные из PostgreSQL для проверки
    df_check = spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", postgres_table) \
        .option("user", postgres_user) \
        .option("password", postgres_password) \
        .option("driver", jdbc_driver) \
        .load()
    
    print(f"Data verification from PostgreSQL:")
    print(f"Total records in table: {df_check.count()}")
    print(f"\nTable schema:")
    df_check.printSchema()
    print(f"\nFirst 5 rows:")
    df_check.show(5, truncate=False)
    
except Exception as e:
    print(f"Error during verification: {str(e)}")
    raise

Data verification from PostgreSQL:
Total records in table: 9800

Table schema:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)


First 5 rows:
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+-----------